# CareLoop Clinics — Part 4
GenAI follow-up note analysis. The API key is entered at runtime and must never be saved.

In [ ]:
!pip install -q google-genai pydantic
import json, time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

MODEL_NAME = "YOUR_FLASH_MODEL_NAME"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

ALLOWED = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
}

API_KEY = input("Paste your Gemini API key at runtime: ")
client = genai.Client(api_key=API_KEY)


## Task 4.1 — validated call

In [ ]:
SYSTEM_TEXT = (
    "You classify outpatient appointment follow-up notes. "
    "Return only the requested JSON structure. Never invent facts."
)

def analyse_note(note_text, prompt_version="CRAFT"):
    prompt = f"""
Context: CareLoop Clinics reviews short call-centre notes about missed appointments.
Role: You are a careful healthcare operations analyst.
Action: Classify the note into exactly one allowed root cause, give a confidence from 0 to 1,
summarize it in at most 25 words, and recommend a practical follow-up action.
Tone: concise, neutral, evidence-based.
Format: JSON only with root_cause, confidence, summary, recommended_action.

Allowed root_cause values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other

If the note is ambiguous, choose other rather than guessing.

Note:
{note_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            # If your current Flash model rejects thinking_config, remove this line.
            thinking_config=types.ThinkingConfig(thinking_level="minimal")
        )
    )
    return json.loads(response.text)

print(analyse_note(pd.read_csv("data/followup_notes.csv").iloc[0]["note_text"]))


## Task 4.2 — Pydantic validation and 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def validate_root_cause(value):
    if value not in ALLOWED:
        raise ValueError(f"root_cause must be one of: {sorted(ALLOWED)}")

notes = pd.read_csv("data/followup_notes.csv").head(10)
results = []

for _, row in notes.iterrows():
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    raw_response = ""
    output = {}
    status = "invalid"

    try:
        output = analyse_note(str(note_text).strip())
        raw_response = json.dumps(output, ensure_ascii=False)
        insight = FollowupInsight(**output)
        validate_root_cause(insight.root_cause)
        status = "valid"
    except Exception as exc:
        print(f"{row['note_id']} validation error:", exc)

    results.append({
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": output.get("root_cause", ""),
        "confidence": output.get("confidence", ""),
        "summary": output.get("summary", ""),
        "recommended_action": output.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_response
    })

    time.sleep(13)

notes_analysed = pd.DataFrame(results, columns=[
    "note_id","appointment_id","root_cause","confidence",
    "summary","recommended_action","validation_status","raw_response"
])
notes_analysed.to_csv("notes_analysed.csv", index=False)
print(notes_analysed)


## Task 4.3 — deliberately bad replies and prompt comparison

In [ ]:
bad_replies = [
    {
        "root_cause": "other",
        "confidence": 1.7,
        "summary": "Invalid confidence",
        "recommended_action": "Re-call patient"
    },
    {
        "root_cause": "other",
        "confidence": 0.5,
        "summary": "Missing action"
    }
]

for bad in bad_replies:
    try:
        FollowupInsight(**bad)
    except ValidationError as exc:
        print(exc)

# Compare V1 and V2 on N007 and N009.
notes_lookup = pd.read_csv("data/followup_notes.csv").set_index("note_id")
comparison = []

for note_id in ["N007", "N009"]:
    note = notes_lookup.loc[note_id, "note_text"]

    v1_prompt = f"Classify this note.\n\n{note}"
    v2_prompt = f"""
Context: CareLoop Clinics classifies missed-appointment call notes.
Role: Careful operations analyst.
Action: Select exactly one allowed root cause. If ambiguous, use other rather than guessing.
Tone: concise and evidence-based.
Format: JSON only.

Allowed values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other

Note:
{note}
"""

    for version, prompt in [("V1", v1_prompt), ("V2", v2_prompt)]:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                max_output_tokens=1000,
                response_mime_type="application/json",
                response_schema=SCHEMA
            )
        )
        data = json.loads(response.text)
        comparison.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": data.get("root_cause", ""),
            "allowed": data.get("root_cause", "") in ALLOWED
        })
        time.sleep(13)

comparison_df = pd.DataFrame(comparison)
print(comparison_df)
print("Deploy V2 because it explicitly constrains the allowed labels and instructs the model to use other when evidence is insufficient.")


## Task 4.4 — merge, crosstab and recommendation inputs

In [ ]:
appointments = pd.read_csv("appointments_clean.csv")
notes_analysed = pd.read_csv("notes_analysed.csv")

merged = notes_analysed.merge(appointments, on="appointment_id", how="left")
print("Merged rows:", len(merged))
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))

# Before submission:
# 1. Verify notes_analysed.csv has 10 rows and 8 columns.
# 2. Remove/clear the API key from notebook outputs.
# 3. Search the notebook for the first characters of your key.
# 4. Put your public GitHub repository URL at the top of README.md.
